# Walkthrough: the refusal direction

This replicates the main steps of Arditi et al. (2024) on a small model: collect activations, find the direction, check it causally (by ablating and by adding it), and confirm that the weight edit matches the hook.

Only aggregate refusal rates are shown for harmful prompts.

In [ ]:
from refusal_lab.config import LabConfig
from refusal_lab.data import load_harmful, load_harmless, split_prompts
from refusal_lab.model import load_model

cfg = LabConfig(n_train=32, n_val=16, max_new_tokens=32)
model, tok = load_model(cfg)
harm_tr, harm_val = split_prompts(load_harmful(), cfg.n_train, cfg.n_val)
ok_tr, ok_val = split_prompts(load_harmless(), cfg.n_train, cfg.n_val)

## 1. Activations and the difference-in-means direction

In [ ]:
from refusal_lab.activations import collect_residuals
from refusal_lab.direction import select_direction

A_harm = collect_residuals(model, tok, harm_tr, cfg)
A_ok = collect_residuals(model, tok, ok_tr, cfg)
best, directions, scores = select_direction(A_harm, A_ok, cfg)
best

In [ ]:
import matplotlib.pyplot as plt

plt.plot(scores[:, -1])
plt.axvline(best.layer, ls="--", c="k")
plt.xlabel("layer")
plt.ylabel("Cohen's d")
plt.title("Separation at last token");

## 2. Projection histogram
How far apart are the two prompt sets along the chosen direction?

In [ ]:
p_harm = A_harm[:, best.layer, -1] @ best.vector
p_ok = A_ok[:, best.layer, -1] @ best.vector
plt.hist(p_ok, alpha=0.6, label="harmless")
plt.hist(p_harm, alpha=0.6, label="harmful")
plt.legend();

## 3. Causal tests
**Ablation** should lower the refusal rate on harmful prompts. **Addition** should raise it on harmless prompts.

In [ ]:
from refusal_lab.evaluate import refusal_rate
from refusal_lab.generate import generate
from refusal_lab.intervene import ablate_direction, add_direction

print("baseline harmful refusal:", refusal_rate(generate(model, tok, harm_val, cfg)))
with ablate_direction(model, best.vector):
    print("ablated  harmful refusal:", refusal_rate(generate(model, tok, harm_val, cfg)))
with add_direction(model, best.vector, best.layer, coeff=float(p_harm.mean())):
    print("added    harmless refusal:", refusal_rate(generate(model, tok, ok_val, cfg)))

## 4. What it costs: KL on harmless prompts

In [ ]:
from refusal_lab.evaluate import mean_kl, next_token_logprobs

base = next_token_logprobs(model, tok, ok_val, cfg)
with ablate_direction(model, best.vector):
    abl = next_token_logprobs(model, tok, ok_val, cfg)
mean_kl(base, abl)

## Exercises
1. Pick a layer far from the selected one and ablate its direction instead. What happens to the refusal rate?
2. Compare the selected directions across two model sizes using cosine similarity of the projections.
3. Swap the keyword detector for a classifier (e.g. LlamaGuard) and see how much the measured refusal rate changes.